# Fabric Canonical Model: component demo

Ejemplo autocontenido con datos sintéticos. Ejecuta los componentes de mapping, resolución, transformación y calidad sin leer ni escribir tablas del Lakehouse. Requiere que el paquete `fabric-canonical-model-framework` esté instalado en el Environment adjunto al notebook.

In [ ]:
from fabric_canonical_model.lib_mapping import AttributeMapper
from fabric_canonical_model.lib_resolution import EntityResolver
from fabric_canonical_model.lib_transformations import TransformationEngine
from fabric_canonical_model.lib_validation import DataValidator
from fabric_canonical_model.lib_utils import get_spark

spark = get_spark()

## 1. Crear fuentes sintéticas

Las fuentes usan nombres físicos distintos para `Well_Name`; `UWI` funciona como clave común.

In [ ]:
source_dataframes = {
    "AVM": spark.createDataFrame(
        [("UWI-001", "100", "  Pozo Norte  "), ("UWI-002", "200", None)],
        ["UWI", "WELL_ID", "WELL_NAME"]
    ),
    "BDP": spark.createDataFrame(
        [("UWI-001", "100", "Pozo N-1"), ("UWI-002", "200", "Pozo Sur")],
        ["UWI", "WELL_ID", "NOMBRE_POZO"]
    ),
    "OFM": spark.createDataFrame(
        [("UWI-001", "100", "Norte 1"), ("UWI-002", "200", None)],
        ["UWI", "WELL_ID", "WELL_DESCRIPTION"]
    )
}

attributes = [
    {
        "name": "Well_ID",
        "type": "string",
        "mapping": {
            "AVM": "WELL_ID",
            "BDP": "WELL_ID",
            "OFM": "WELL_ID"
        },
        "strategy": "CONSISTENT",
        "priority": ["AVM", "BDP", "OFM"]
    },
    {
        "name": "Well_Name",
        "type": "string",
        "mapping": {
            "AVM": "WELL_NAME",
            "BDP": "NOMBRE_POZO",
            "OFM": "WELL_DESCRIPTION"
        },
        "strategy": "SOURCE_PRIORITY",
        "priority": ["AVM", "BDP", "OFM"]
    }
]

## 2. Mapear y resolver

`CONSISTENT` detiene el procesamiento si `Well_ID` difiere entre fuentes. `SOURCE_PRIORITY` elige el primer valor no nulo para `Well_Name`.

In [ ]:
mapped_dataframes = AttributeMapper().map_sources(
    dataframes=source_dataframes,
    attributes_config=attributes,
    include_unmapped=True
)

canonical_df = EntityResolver().resolve_entity(
    dataframes=mapped_dataframes,
    business_keys=["UWI"],
    attributes_config=attributes
)

canonical_df.show(truncate=False)

## 3. Transformar y validar

El ejemplo aplica transformaciones declarativas y comprueba claves, atributos requeridos y duplicados.

In [ ]:
canonical_df = TransformationEngine().apply(
    canonical_df,
    [
        {"operation": "TRIM", "column": "UWI"},
        {"operation": "TRIM", "column": "Well_Name"},
        {"operation": "INITCAP", "column": "Well_Name"}
    ]
)

quality = DataValidator().validate_dataframe(
    dataframe=canonical_df,
    business_keys=["UWI"],
    required_columns=["Well_ID", "Well_Name"],
    check_duplicates=True
)

if not quality["passed"]:
    raise ValueError(f"La validación de calidad falló: {quality}")

canonical_df.orderBy("UWI").show(truncate=False)
quality

Este notebook es una demostración de componentes con datos sintéticos; no ejecuta auditoría, watermarks ni persistencia Delta. Para procesar entidades del Lakehouse, usa `notebooks/nb_create_canonical_model.ipynb` después de instalar el paquete y desplegar las configuraciones YAML del proyecto.